# Task note: gesture recognition and split mode

The label is the gesture, parsed from the filename (`gestureN`).

`SPLIT_MODE` (set in the split cell) controls how files are divided:

- `'grouped'` (default): `GroupShuffleSplit` by **participant ID**, so no participant appears in more than one of the splits. This measures generalisation to unseen users.
- `'random'`: the previous behaviour, a random per-file split. The same participants appear in every split, so the numbers are within-subject.

**Participant ID from the filename.** `extract_group_from_filename` splits the basename on `_` and takes the token containing `participant` (e.g. `participant7` -> 7). This scheme was verified from the code only: the same parsing is used in `folders_creation.ipynb` (including the cell that builds `17participants`, and the `session1_participantN` / `session3_participantN` folder globs) and in the `*part*` notebooks. The real files are not available here, so it was not checked against actual data; the split cell asserts that every file yields an ID and stops with an error otherwise. The other filename tokens (session, trial) are not used.

Caveat: the `s1_s2_s3` folders mix sessions. Grouping by participant number assumes the same number means the same person in every session; this cannot be verified from the code. If it is wrong, change the parser.

Status: the code in this notebook has been changed (see the README "Known limitations") but has **not been re-run**; numbers from earlier runs are not comparable.

In [ ]:
# Reproducibility: one seed for every source of randomness used below.
import os
import random
import numpy as np
import torch

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# DataLoader seeding: every DataLoader below gets worker_init_fn=seed_worker and generator=data_generator.
def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

data_generator = torch.Generator()
data_generator.manual_seed(SEED)
# Note: bit-exact GPU reproducibility is not guaranteed (non-deterministic CUDA kernels, DataParallel).

In [ ]:
import os
import numpy as np
import scipy.io as sio
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import ViTForImageClassification, ViTFeatureExtractor
from tqdm import tqdm
import glob

# Function to extract the participant ID from the filename
# def extract_participant_from_filename(filepath):
#     filename = os.path.basename(filepath)
#     parts = filename.split('_')
#     for part in parts:
#         if 'participant' in part:
#             participant_id = part.replace('participant', '')
#             return int(participant_id)  # Convert to integer
#     return None  # If no participant ID found

# Function to extract gesture label from the filename
def extract_label_from_filename(filepath):
    filename = os.path.basename(filepath)
    parts = filename.split('_')
    for part in parts:
        if 'gesture' in part:
            return int(part.replace('gesture', ''))
    return None

# Function to load and process a single .mat file
def load_and_process_file(filepath, target_length):
    mat_data = sio.loadmat(filepath)
    data_emg = mat_data['data_emg']
    
    # Handle NaNs and Infinities
    data_emg = np.nan_to_num(data_emg, nan=0.0, posinf=0.0, neginf=0.0)
    
    # Pad or truncate the data to the target length
    if data_emg.shape[0] < target_length:
        padded_data = np.pad(data_emg, ((0, target_length - data_emg.shape[0]), (0, 0)), 'constant', constant_values=0)
    else:
        padded_data = data_emg[:target_length]

    # Reshape to 224x224x3 required by ViT (ensure the size is consistent)
    padded_data = padded_data.flatten()  # Flatten before reshaping
    num_pixels = 3 * 224 * 224 
    
    # If we don't have enough data, we can pad with zeros; otherwise, truncate
    if padded_data.size < num_pixels:
        reshaped_data = np.pad(padded_data, (0, num_pixels - padded_data.size), 'constant', constant_values=0)
    else:
        reshaped_data = padded_data[:num_pixels]
    
    # Reshape to (224, 224, 3) as required by ViT
    reshaped_data = np.reshape(reshaped_data, (3, 224, 224))
    
    # No per-sample min-max here: per-channel mean/std (training data only) is applied in EMGDataset.

    return reshaped_data

In [ ]:
# Per-channel mean/std, computed on the TRAINING files only and applied to train / val / test.
# "Channel" = axis 0 of the (3, 224, 224) array that load_and_process_file builds from the flattened EMG window.
def compute_channel_stats(file_list, target_length, eps=1e-8):
    n = 0
    s = np.zeros(3, dtype=np.float64)
    ss = np.zeros(3, dtype=np.float64)
    for f in tqdm(file_list, desc='channel stats (train only)'):
        x = load_and_process_file(f, target_length).astype(np.float64)
        s += x.sum(axis=(1, 2))
        ss += (x ** 2).sum(axis=(1, 2))
        n += x.shape[1] * x.shape[2]
    mean = s / n
    std = np.sqrt(np.maximum(ss / n - mean ** 2, 0.0))
    std = np.where(std < eps, 1.0, std)   # guard against zero std
    return mean, std

# Custom dataset for the EMG data
class EMGDataset(Dataset):
    def __init__(self, file_list, target_length, labels, norm_stats=None):
        self.file_list = file_list
        self.target_length = target_length
        self.labels = labels
        self.norm_stats = norm_stats  # (mean, std), each of shape (3,), fitted on the training split only

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        file_path = self.file_list[idx]
        label = self.labels[idx]
        emg_image = load_and_process_file(file_path, self.target_length)
        if self.norm_stats is not None:
            mean, std = self.norm_stats
            emg_image = (emg_image - mean[:, None, None]) / std[:, None, None]
        return emg_image.astype(np.float32), label

# Path to the root folder where all .mat files are stored
root_folder = 's1_s2_s3_matfiles_10gest'  # Update with your folder path

# Find all .mat files in the folder
all_mat_files = glob.glob(os.path.join(root_folder, '*.mat'))

# Collect all corresponding labels
labels = [extract_label_from_filename(mat_file) for mat_file in all_mat_files]

# Original unique labels
unique_labels = np.unique(labels)

# Create a mapping from the original labels to the new range 0-4
label_mapping = {original_label: new_label for new_label, original_label in enumerate(unique_labels)}

# Convert the original labels to the new range using the mapping
converted_labels = np.array([label_mapping[label] for label in labels])

labels = converted_labels
# Target length for EMG signals (adjust based on your data)
target_length = 10240  # Modify this as needed

# ---- Train / validation / test split ----
# The test set is held out and evaluated ONCE after training; checkpoint selection uses the validation split,
# which is carved out of the TRAIN portion.
TEST_SIZE = 0.2
VAL_SIZE = 0.2   # fraction of the train+val portion
SPLIT_MODE = 'grouped'  # 'grouped' = participant-grouped split (default), 'random' = previous random per-file split
from sklearn.model_selection import GroupShuffleSplit

# Participant ID from the filename (same scheme as folders_creation.ipynb / the *part* notebooks):
# the basename is split on '_' and the token containing 'participant' is used, e.g. 'participant7' -> 7.
def extract_group_from_filename(filepath):
    filename = os.path.basename(filepath)
    for part in filename.split('_'):
        if 'participant' in part:
            return int(part.replace('participant', ''))
    return None

all_mat_files = np.array(all_mat_files)
labels = np.asarray(labels)
idx = np.arange(len(all_mat_files))

if SPLIT_MODE == 'grouped':
    groups = np.array([extract_group_from_filename(f) for f in all_mat_files], dtype=object)
    assert all(g is not None for g in groups), \
        "Could not parse a participant ID from every filename; fix extract_group_from_filename or use SPLIT_MODE='random'."
    trainval_idx, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SEED).split(idx, labels, groups))
    rel_tr, rel_va = next(GroupShuffleSplit(n_splits=1, test_size=VAL_SIZE, random_state=SEED).split(
        trainval_idx, labels[trainval_idx], groups[trainval_idx]))
    train_idx, val_idx = trainval_idx[rel_tr], trainval_idx[rel_va]
    g_tr, g_va, g_te = set(groups[train_idx]), set(groups[val_idx]), set(groups[test_idx])
    assert not (g_tr & g_va) and not (g_tr & g_te) and not (g_va & g_te), 'participant leakage between splits'
    print(f"Grouped split: participants train={len(g_tr)}, val={len(g_va)}, test={len(g_te)}")
elif SPLIT_MODE == 'random':
    trainval_idx, test_idx = train_test_split(idx, test_size=TEST_SIZE, random_state=SEED)
    train_idx, val_idx = train_test_split(trainval_idx, test_size=VAL_SIZE, random_state=SEED)
else:
    raise ValueError("SPLIT_MODE must be 'grouped' or 'random'")

train_files, val_files, test_files = [list(all_mat_files[i]) for i in (train_idx, val_idx, test_idx)]
train_labels, val_labels, test_labels = labels[train_idx], labels[val_idx], labels[test_idx]
print(f"Files: train={len(train_files)}, val={len(val_files)}, test={len(test_files)}")

# Normalisation statistics from the training split only
channel_mean, channel_std = compute_channel_stats(train_files, target_length)
norm_stats = (channel_mean, channel_std)
print('Channel mean:', channel_mean, 'std:', channel_std)

# Create datasets and dataloaders
train_dataset = EMGDataset(train_files, target_length, train_labels, norm_stats=norm_stats)
val_dataset = EMGDataset(val_files, target_length, val_labels, norm_stats=norm_stats)
test_dataset = EMGDataset(test_files, target_length, test_labels, norm_stats=norm_stats)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, generator=data_generator, worker_init_fn=seed_worker)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, worker_init_fn=seed_worker)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, worker_init_fn=seed_worker)

In [ ]:
print(np.unique(labels))
labels.dtype

In [ ]:
# Load a pre-trained ViT model from Hugging Face
feature_extractor = ViTFeatureExtractor.from_pretrained('google/vit-base-patch16-224-in21k')
model = ViTForImageClassification.from_pretrained('google/vit-base-patch16-224-in21k', num_labels=10)  # Adjust num_labels as needed

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Define optimizer and loss function
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
criterion = torch.nn.CrossEntropyLoss()

In [ ]:
# --- Training loop for 50 epochs ---
import mlflow
import mlflow.pytorch
num_epochs = 50

# Track experiment with MLflow
def start_mlflow_experiment(experiment_name):
    mlflow.set_experiment(experiment_name)
    mlflow.start_run()

def end_mlflow_experiment():
    mlflow.end_run()

# Define the name of the experiment based on the input file or another identifier
def get_experiment_name_from_file(filepath):
    experiment_name = os.path.basename(filepath).split('.')[0]  # Extract filename without extension
    return experiment_name

# Initialize MLflow experiment (This can be placed at the start of your main function)
experiment_name = get_experiment_name_from_file(root_folder)  # Using folder as experiment name
start_mlflow_experiment(experiment_name)

best_val_acc = 0.0  # checkpoint selection uses the VALIDATION split only
best_model_path = f'best_model_{experiment_name}.pth'

for epoch in range(num_epochs):
    # ---- Training ----
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for emg_data, targets in tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}'):
        emg_data = emg_data.to(device)
        targets = targets.to(device)
        # Prepare input for ViT by treating EMG data as image-like input
        inputs = feature_extractor(emg_data, return_tensors="pt", do_rescale=False)['pixel_values'].to(device)

        # Forward pass
        outputs = model(pixel_values=inputs)
        loss = criterion(outputs.logits, targets)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Accumulate loss and accuracy over the whole epoch
        _, predicted = torch.max(outputs.logits, 1)
        total += targets.size(0)
        correct += (predicted == targets).sum().item()
        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)   # mean over batches
    train_acc = correct / total                      # over the whole epoch
    mlflow.log_metric("train_loss", train_loss, step=epoch)
    mlflow.log_metric("train_accuracy", train_acc, step=epoch)
    print(f'Epoch {epoch+1}, Loss: {train_loss:.4f}, Accuracy: {train_acc:.4f}')

    # ---- Validation (validation split carved from the train data; the test set is NOT touched here) ----
    model.eval()
    correct = 0
    total = 0
    val_loss_sum = 0.0
    with torch.no_grad():
        for emg_data, targets in val_loader:
            emg_data = emg_data.to(device)
            targets = targets.to(device)

            inputs = feature_extractor(emg_data, return_tensors="pt", do_rescale=False)['pixel_values'].to(device)
            outputs = model(pixel_values=inputs)
            batch_loss = criterion(outputs.logits, targets)   # loss of THIS validation batch
            val_loss_sum += batch_loss.item()
            _, predicted = torch.max(outputs.logits, 1)
            total += targets.size(0)
            correct += (predicted == targets).sum().item()
    val_loss = val_loss_sum / len(val_loader)   # mean over batches
    val_acc = correct / total
    print(f'Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_acc:.4f}')
    mlflow.log_metric("val_loss", val_loss, step=epoch)
    mlflow.log_metric("val_accuracy", val_acc, step=epoch)
    mlflow.pytorch.log_model(model, "models/last_model")

    # Keep the checkpoint with the best validation accuracy
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)
        print(f'New best model saved (val accuracy {best_val_acc:.4f})')
end_mlflow_experiment()

In [ ]:
# Final evaluation: the TEST set is used once, with the checkpoint selected on validation accuracy.
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()
y_true_all, y_pred_all = [], []
with torch.no_grad():
    for emg_data, targets in test_loader:
        emg_data = emg_data.to(device)
        inputs = feature_extractor(emg_data, return_tensors="pt", do_rescale=False)['pixel_values'].to(device)
        outputs = model(pixel_values=inputs)
        y_pred_all.append(outputs.logits.argmax(dim=1).cpu().numpy())
        y_true_all.append(targets.numpy())
y_true_all = np.concatenate(y_true_all)
y_pred_all = np.concatenate(y_pred_all)

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

def report_eval(y_true, y_pred, labels, names=None, title='Test set'):
    """Accuracy, chance level, per-class accuracy and confusion matrix."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    labels = list(labels)
    names = [str(n) for n in (labels if names is None else names)]
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    support = cm.sum(axis=1)
    per_class = np.divide(cm.diagonal(), support, out=np.full(len(labels), np.nan), where=support > 0)
    acc = float((y_true == y_pred).mean())
    chance_uniform = 1.0 / len(labels)
    chance_majority = support.max() / support.sum()
    print(f'{title} accuracy: {acc:.4f}')
    print(f'Chance level: uniform 1/K = {chance_uniform:.4f} (K={len(labels)}), majority class = {chance_majority:.4f}')
    print('Per-class accuracy:')
    for n, a, s in zip(names, per_class, support):
        print(f'  class {n}: {a:.3f}  (n={s})')
    fig, axes = plt.subplots(1, 2, figsize=(max(10, 0.6 * len(labels) + 8), max(4, 0.35 * len(labels) + 3)))
    ConfusionMatrixDisplay(cm, display_labels=names).plot(ax=axes[0], colorbar=False, xticks_rotation=90)
    axes[0].set_title(f'{title}: confusion matrix')
    axes[1].bar(names, per_class)
    axes[1].axhline(chance_uniform, color='r', linestyle='--', label='uniform chance')
    axes[1].axhline(chance_majority, color='k', linestyle=':', label='majority-class chance')
    axes[1].set_ylim(0, 1)
    axes[1].set_xlabel('class')
    axes[1].set_ylabel('accuracy')
    axes[1].set_title(f'{title}: per-class accuracy')
    axes[1].legend()
    plt.setp(axes[1].get_xticklabels(), rotation=90)
    plt.tight_layout()
    plt.show()
    return acc, cm, per_class

test_acc, test_cm, test_per_class = report_eval(
    y_true_all, y_pred_all, labels=np.arange(len(unique_labels)), names=unique_labels, title='Test set (ViT)')